# 05 — Embedding cache

Every production LLM pipeline eventually pays to memoize its embeddings — user queries repeat, RAG chunks are re-ingested, regressions re-run the same prompts. `CachingEmbeddingModel` is a drop-in decorator around any LangChain4j `EmbeddingModel` that turns the wrapped model into a cache-aware one: identical text resolves from the cache; everything else goes to the delegate.

This notebook measures the hit-rate and delegate-call count across three phases — a cold batch, a mostly-hit batch, and a single-text embed on a hot key.

In [1]:
%classpath /home/jovyan/work/vectors/vectors-core/build/libs/vectors-core-0.1.0-SNAPSHOT.jar
%classpath /home/jovyan/work/vectors/vectors-cache/build/libs/vectors-cache-0.1.0-SNAPSHOT.jar
%classpath /home/jovyan/work/vectors/vectors-cache-langchain4j/build/libs/vectors-cache-langchain4j-0.1.0-SNAPSHOT.jar


In [2]:
%%loadFromPOM
<dependencies>
  <dependency>
    <groupId>dev.langchain4j</groupId>
    <artifactId>langchain4j-core</artifactId>
    <version>1.0.0-beta1</version>
  </dependency>
  <dependency>
    <groupId>com.github.ben-manes.caffeine</groupId>
    <artifactId>caffeine</artifactId>
    <version>3.1.8</version>
  </dependency>
</dependencies>


In [3]:
import com.integrallis.vectors.cache.CaffeineVectorCache;
import com.integrallis.vectors.cache.CacheStats;
import com.integrallis.vectors.cache.VectorCache;
import com.integrallis.vectors.cache.langchain4j.CachingEmbeddingModel;
import dev.langchain4j.data.embedding.Embedding;
import dev.langchain4j.data.segment.TextSegment;
import dev.langchain4j.model.embedding.EmbeddingModel;
import dev.langchain4j.model.output.Response;
import java.time.Duration;
import java.util.*;
import java.util.concurrent.atomic.AtomicInteger;

final AtomicInteger delegateCalls = new AtomicInteger();

final class CountingTrigramEmbedder implements EmbeddingModel {
    private final int dim;
    CountingTrigramEmbedder(int dim) { this.dim = dim; }
    @Override public int dimension() { return dim; }
    @Override public Response<Embedding> embed(String t) {
        delegateCalls.incrementAndGet();
        return Response.from(Embedding.from(trigram(t)));
    }
    @Override public Response<Embedding> embed(TextSegment s) { return embed(s.text()); }
    @Override public Response<List<Embedding>> embedAll(List<TextSegment> segs) {
        List<Embedding> out = new ArrayList<>(segs.size());
        for (TextSegment s : segs) { delegateCalls.incrementAndGet(); out.add(Embedding.from(trigram(s.text()))); }
        return Response.from(out);
    }
    private float[] trigram(String t) {
        float[] v = new float[dim];
        String s = t.toLowerCase(); if (s.length() < 3) s = "  " + s + "  ";
        for (int i = 0; i <= s.length()-3; i++) { int h = 31*(31*s.charAt(i)+s.charAt(i+1))+s.charAt(i+2); v[Math.floorMod(h, dim)] += 1f; }
        float n = 0f; for (float x : v) n += x*x; n = (float)Math.sqrt(n);
        if (n > 0) for (int i = 0; i < dim; i++) v[i] /= n;
        return v;
    }
}

In [4]:
VectorCache<String, float[]> cache = CaffeineVectorCache.<String, float[]>builder()
    .maximumSize(10_000L)
    .expireAfterWrite(Duration.ofHours(1))
    .build();

EmbeddingModel raw = new CountingTrigramEmbedder(128);
EmbeddingModel cached = new CachingEmbeddingModel(raw, cache);

void report(String label) {
    CacheStats s = cache.stats();
    System.out.printf("%-22s hits=%d misses=%d size=%d hitRate=%.1f%%  delegateCalls=%d%n",
        label, s.hits(), s.misses(), s.size(), 100.0 * s.hitRate(), delegateCalls.get());
}

void embedBatch(List<String> texts) {
    List<TextSegment> segs = texts.stream().map(TextSegment::from).toList();
    cached.embedAll(segs);
}

embedBatch(List.of(
    "What is HNSW?",
    "Explain product quantization.",
    "How does mmap persistence work?",
    "What is HNSW?",
    "Describe scalar quantization.",
    "Explain product quantization."));
report("after phase 1");

embedBatch(List.of(
    "What is HNSW?",
    "Explain product quantization.",
    "What is ANN?"));
report("after phase 2");

cached.embed("What is HNSW?");
report("after phase 3");

cache.close();

after phase 1

 hits=

0

 misses=

6

...

 hitRate=

30.0

%

  delegateCalls=

7

## What to notice

- **Phase 1** is cold — six new texts, all miss. Because `CachingEmbeddingModel.embedAll` coalesces misses into a single delegate round-trip, there is one `embedAll` call but six per-text invocations of the underlying `trigram()`. Four unique keys persist.
- **Phase 2** issues three texts, two already seen — two cache hits, one new miss. Only one delegate call.
- **Phase 3** single-text hot-key path (`embed(String)`) returns from the cache without touching the delegate.

Swap `CaffeineVectorCache` for `JCacheVectorCache` (JSR-107 bridge) or any other `VectorCache` implementation when you want to share the cache across JVMs.